# 03) Baseline Models and Feature Selection

In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

train = pd.read_csv("data/train.csv", parse_dates=["date"])
with open("data/columns.json") as f:
    columns = json.load(f)

TARGET_COL = columns["target_cols"][0]
FEATURE_COLS = columns["feature_cols"]

print(f"train: {train.shape[0]} rows, {len(FEATURE_COLS)} features, target = {TARGET_COL!r}")

train: 8443 rows, 33 features, target = 'target_bps_vs_mid'


## Train / eval split

`data/test.csv` stays untouched until final validation (per the project workflow) -- it is not
touched here. This split carves an eval set out of `data/train.csv` itself, for tuning.

Same rule as the train/test split in `02)`: split by date, not randomly, snapped to a full trading
day at the ~80% cumulative-row-count mark, so no day's rows land on both sides.

In [2]:
daily_counts = train.groupby("date").size().sort_index()
cum_frac = daily_counts.cumsum() / daily_counts.sum()
split_date = cum_frac[cum_frac >= 0.80].index[0]

fit = train[train["date"] <= split_date].copy()
eval_ = train[train["date"] > split_date].copy()

print(f"split date: {split_date.date()} (last day included in fit)")
print(f"fit:  {fit.shape[0]} rows ({fit.shape[0]/len(train):.1%}), "
      f"{fit['date'].nunique()} days, {fit['date'].min().date()} to {fit['date'].max().date()}")
print(f"eval: {eval_.shape[0]} rows ({eval_.shape[0]/len(train):.1%}), "
      f"{eval_['date'].nunique()} days, {eval_['date'].min().date()} to {eval_['date'].max().date()}")
print()
print("no date overlap:", set(fit["date"]).isdisjoint(set(eval_["date"])))

X_fit, y_fit = fit[FEATURE_COLS], fit[TARGET_COL]
X_eval, y_eval = eval_[FEATURE_COLS], eval_[TARGET_COL]

split date: 2025-03-20 (last day included in fit)
fit:  6952 rows (82.3%), 14 days, 2025-03-03 to 2025-03-20
eval: 1491 rows (17.7%), 3 days, 2025-03-21 to 2025-03-25

no date overlap: True


## Ridge regression

Pipeline: median-impute (only `near_far_spread_bps` has nulls, from the symbol-days with no
standalone auction-only orders) -> standardize -> Ridge. Both imputer and scaler are fit on `fit`
only, inside the pipeline, so no statistic from `eval` leaks into preprocessing.

Sweep penalties on a log scale, wide enough to bracket where eval MSE actually bottoms out --
report fit/eval MSE at each.

In [3]:
alphas = [0.1, 1.0, 10.0, 100.0, 1_000.0, 10_000.0, 100_000.0, 1_000_000.0]
results = []

for alpha in alphas:
    model = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("ridge", Ridge(alpha=alpha)),
    ])
    model.fit(X_fit, y_fit)

    fit_mse = mean_squared_error(y_fit, model.predict(X_fit))
    eval_mse = mean_squared_error(y_eval, model.predict(X_eval))
    results.append({"alpha": alpha, "fit_mse": fit_mse, "eval_mse": eval_mse})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

    alpha   fit_mse   eval_mse
      0.1 42.030542  71.564930
      1.0 42.074216  71.569254
     10.0 42.101411  71.370866
    100.0 42.122408  69.947956
   1000.0 42.552607  61.649924
  10000.0 48.884861  59.808235
 100000.0 64.348533  91.665772
1000000.0 69.289679 103.043808
